### 1. 📥​ Imports

In [1]:
%pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, os
from pathlib import Path
import pandas as pd

SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))

from tcibench.preparation import (Dataset, DatasetMetadata, load_dataset, load_metadata)

from tcibench.generation import (
    generate_with_synthcity,
    balance_data
)

/home/labian-3/miniconda3/envs/tdallm-synthcity/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Disabling PyTorch because PyTorch >= 2.5 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


### 2. ⚙️​ Configs

In [3]:
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "tcibench" / "preparation" / "metadata.json").is_file()),
    None,
)

In [4]:
# Reproducibility
SPLIT_SEED = 42
SAMPLING_SEED = 42
GENERATION_SEED = 42

# Generation protocol
EXAMPLES_PER_CLASS = 10
RECORDS_PER_BATCH = 10

📂​ Datasets

In [5]:
DATASETS = {
    "diabetes": "Diabetes",
    "sick": "Thyroid Sick",
}

RAW_ROOT = PROJECT_ROOT / "data" / "raw"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / f"split_seed_{SPLIT_SEED}"

RAW_PATHS = {
    "diabetes": RAW_ROOT / "diabetes.csv",
    "sick": RAW_ROOT / "sick.arff",
}

🤖 Models

In [6]:
#GENERATIVE_MODELS = ["ddpm", "ctgan", "tvae"]
GENERATIVE_MODELS = ["ddpm"]
CLASSICAL_METHODS = ["ros", "rus", "smotenc"]

### 3. 📂​ Dataset

In [7]:
def inspect_dataset(
    data: pd.DataFrame,
    metadata: DatasetMetadata,
    cleaned: pd.DataFrame,
    cleaning_stats: dict,
):
    """Inspect a dataset before running the experiment."""

    print(f"Dataset: {metadata.display_name}")
    print(f"Target: {metadata.target_column}")

    print("\nRaw data")
    print(f"Shape: {data.shape}")
    display(data.head())

    print("\nCleaning")
    print(f"Shape after cleaning: {cleaned.shape}")
    print(f"Removed columns: {cleaning_stats['dropped_columns']}")
    print(f"Removed rows with missing values: {cleaning_stats['removed_missing_rows']}")
    print(f"Removed duplicate rows: {cleaning_stats['removed_duplicate_rows']}")

    print("\nCleaned data")
    display(cleaned.head())

    print("\nTarget distribution")
    target = metadata.target_column
    display(pd.DataFrame({
        "count": cleaned[target].value_counts(),
        "proportion": cleaned[target].value_counts(normalize=True).mul(100).round(2),
    }))

    target_counts = cleaned[target].value_counts()
    print(f"Imbalance ratio: {target_counts.max() / target_counts.min():.2f}")

In [8]:
DATASET_ID = "sick"

metadata = load_metadata(DATASET_ID)
data = load_dataset(RAW_PATHS[DATASET_ID])

dataset = Dataset(data, metadata)

cleaned, cleaning_report = dataset.preprocessing.clean(data)
folds = dataset.preprocessing.split(cleaned)

inspect_dataset(data, metadata, cleaned, cleaning_report)

train, test, audit = folds[0]

counts = train[metadata.target_column].value_counts()
majority_class = counts.idxmax()
minority_class = counts.idxmin()

majority_count = int(counts[majority_class])
minority_count = int(counts[minority_class])
records_to_generate = majority_count - minority_count
imbalance_ratio = majority_count / minority_count

print(f"Dataset: {metadata.display_name}")
print(f"Cleaned: {cleaned.shape}")
print(f"Fold 1 — Train: {train.shape} | Test: {test.shape}")
print(f"Majority: {majority_class} ({majority_count})")
print(f"Minority: {minority_class} ({minority_count})")
print(f"Imbalance ratio: {imbalance_ratio:.2f}")
print(f"Records to generate: {records_to_generate}")

Dataset: Thyroid Sick
Target: Class

Raw data
Shape: (3772, 30)


,age,sex,on_thyroxine,query_on_thyroxine,on_antithyroid_medication,sick,pregnant,thyroid_surgery,I131_treatment,query_hypothyroid,...,TT4_measured,TT4,T4U_measured,T4U,FTI_measured,FTI,TBG_measured,TBG,referral_source,Class
0,41.0,F,f,f,f,f,f,f,f,f,...,t,125.0,t,1.14,t,109.0,f,NaN,SVHC,negative
1,23.0,F,f,f,f,f,f,f,f,f,...,t,102.0,f,NaN,f,NaN,f,NaN,other,negative
2,46.0,M,f,f,f,f,f,f,f,f,...,t,109.0,t,0.91,t,120.0,f,NaN,other,negative
3,70.0,F,t,f,f,f,f,f,f,f,...,t,175.0,f,NaN,f,NaN,f,NaN,other,negative
4,70.0,F,f,f,f,f,f,f,f,f,...,t,61.0,t,0.87,t,70.0,f,NaN,SVI,negative



Cleaning
Shape after cleaning: (2642, 29)
Removed columns: ['TBG']
Removed rows with missing values: 1129
Removed duplicate rows: 1

Cleaned data


,age,sex,on_thyroxine,query_on_thyroxine,on_antithyroid_medication,sick,pregnant,thyroid_surgery,I131_treatment,query_hypothyroid,...,T3,TT4_measured,TT4,T4U_measured,T4U,FTI_measured,FTI,TBG_measured,referral_source,Class
0,41.0,F,f,f,f,f,f,f,f,f,...,2.5,t,125.0,t,1.14,t,109.0,f,SVHC,negative
1,70.0,F,f,f,f,f,f,f,f,f,...,1.2,t,61.0,t,0.87,t,70.0,f,SVI,negative
2,80.0,F,f,f,f,f,f,f,f,f,...,0.6,t,80.0,t,0.70,t,115.0,f,SVI,sick
3,66.0,F,f,f,f,f,f,f,f,f,...,2.2,t,123.0,t,0.93,t,132.0,f,SVI,negative
4,68.0,M,f,f,f,f,f,f,f,f,...,1.6,t,83.0,t,0.89,t,93.0,f,SVI,negative



Target distribution


,count,proportion
Class,,
negative,2430,91.98
sick,212,8.02


Imbalance ratio: 11.46
Dataset: Thyroid Sick
Cleaned: (2642, 29)
Fold 1 — Train: (2113, 29) | Test: (529, 29)
Majority: negative (1944)
Minority: sick (169)
Imbalance ratio: 11.50
Records to generate: 1775


### 4. 🤖 Models generation

In [9]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

import torch
from synthcity.utils.constants import DEVICE

print("PyTorch:", torch.cuda.is_available())
print("Synthcity:", DEVICE)

True
NVIDIA GeForce RTX 4060 Ti
PyTorch: True
Synthcity: cuda


In [10]:
OUTPUT_ROOT = PROJECT_ROOT / "artifacts" / "generation"
report_path = (
    OUTPUT_ROOT / metadata.dataset_id / "baselines" / "generative_stats.json"
)
report_path.parent.mkdir(parents=True, exist_ok=True)

all_stats = []

for fold_id, (train, test, audit) in enumerate(folds):
    counts = train[metadata.target_column].value_counts()
    minority_class = counts.idxmin()
    majority_class = counts.idxmax()

    n_samples = int(
        counts[majority_class] - counts[minority_class]
    )

    for model_name in GENERATIVE_MODELS:
        synthetic, stats = generate_with_synthcity(
            train,
            test=test,
            metadata=metadata,
            model_name=model_name,
            n_samples=n_samples,
            random_seed=GENERATION_SEED,
            target_column=metadata.target_column,
            minority_class=minority_class,
            fold_id=fold_id,
            output_root=OUTPUT_ROOT,
        )

        stats.insert(0, "fold", fold_id)
        all_stats.append(stats)

        # Keep completed runs if a later model fails.
        pd.concat(all_stats, ignore_index=True).to_json(
            report_path, orient="records", indent=2
        )

        print(
            f"Fold {fold_id} | "
            f"Model: {model_name} | "
            f"Minority: {minority_class} | "
            f"Generated: {len(synthetic)}"
        )

        display(stats)

[2026-10-14T14:46:20.741095-0300][522879][CRITICAL] Error importing TabularGoggle: No module named 'dgl'
[2026-10-14T14:46:20.744309-0300][522879][CRITICAL] module disabled: /home/labian-3/miniconda3/envs/tdallm-synthcity/lib/python3.10/site-packages/synthcity/plugins/generic/plugin_goggle.py
Epoch: 100%|██████████| 1000/1000 [01:02<00:00, 15.96it/s, loss=0.551]


ddpm | fold 0 | attempt 1/10 | generated 1953 | wrong class 0 | duplicates 489 | accumulated 0.82  classes {'sick': 1953} | copies of train 0


ValueError: Invalid artifact: 1464 rows, expected 1775

In [ ]:
for fold_id, (train, test, audit) in enumerate(folds):
    for method in CLASSICAL_METHODS:
        actual_method = method
        if method == "smotenc" and not any(
            column in train.columns
            for column in metadata.categorical_columns
        ):
            actual_method = "smote"

        balanced_train, stats = balance_data(
            train,
            test=test,
            metadata=metadata,
            method=method,
            fold_id=fold_id,
            random_seed=GENERATION_SEED,
            output_root=PROJECT_ROOT / "artifacts" / "generation"
        )

        stats.insert(0, "fold", fold_id)
        all_stats.append(stats)

        # Keep one report with generative and classical baseline statistics.
        pd.concat(all_stats, ignore_index=True).to_json(
            report_path, orient="records", indent=2
        )

        print(
            f"Fold {fold_id} | "
            f"Method: {actual_method} | "
            f"Rows saved: {len(balanced_train)}"
        )

FileExistsError: Output already exists: /home/labian-3/Documentos/Fernanda/tcibench/artifacts/generation/sick/baselines/ros/fold_0.csv